In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import scipy
import time
from itertools import combinations

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import SGDOneClassSVM
from sklearn.kernel_approximation import Nystroem
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

## Chargement des données

In [ ]:
filepath = r"C:\Users\melan\DESU_DATA_2026\ProjetNR\data2026.xlsx"
raw_df = pd.read_excel(filepath, engine="openpyxl", sheet_name='Total2', header=0,
                       na_values=['na', 'NaN', 'NA', 'N/A', 'n/a'])
raw_df.head()

## Encodage de la variable categorielle Hipp

On encode Hipp tout au debut, avant le nettoyage et l'imputation, pour que ces etapes
s'appliquent aussi aux colonnes generees (Hipp_NI, Hipp_ZE, Hipp_ZP, ...).

In [ ]:
encoder = OneHotEncoder(sparse_output=False)
hipp_encoded = encoder.fit_transform(raw_df[["Hipp"]])
hipp_columns = list(encoder.get_feature_names_out(["Hipp"]))

hipp_encoded_df = pd.DataFrame(hipp_encoded, columns=hipp_columns, index=raw_df.index)

# On remplace la colonne categorielle Hipp par ses colonnes encodees (0/1)
raw_df = pd.concat([raw_df.drop(columns=["Hipp"]), hipp_encoded_df], axis=1)
raw_df.head()

## Nettoyage des données

In [ ]:
percent_missing = raw_df.isnull().sum() * 100 / len(raw_df)
percent_missing.sort_values(ascending=False, inplace=True)
percent_missing

In [ ]:
threshold_view = 10

filtered = percent_missing[percent_missing.values > threshold_view]
ax = sns.barplot(x=filtered, y=filtered.index, orient='h')
ax.set_title(f"Répartition du pourcentage de valeurs manquantes supérieures au seuil de {threshold_view}")

threshold = 40
ax.axvline(x=threshold, color='r', linestyle='--', label=f"Seuil de {threshold}")

In [ ]:
columns_to_drop = percent_missing[percent_missing.values > threshold].index
columns_to_drop

In [ ]:
raw_df.drop(columns=columns_to_drop, inplace=True, errors="ignore")
raw_df.shape

## Imputation des valeurs manquantes

In [ ]:
numeric_features = raw_df.select_dtypes(include=['float', 'int'])
numeric_features.shape

In [ ]:
imputation = KNNImputer(missing_values=np.nan)
imputed = imputation.fit_transform(numeric_features)
imputed = pd.DataFrame(imputed, columns=numeric_features.columns, index=raw_df.index)
imputed.shape

In [ ]:
raw_df.loc[:, numeric_features.columns] = imputed
numeric_features.columns

In [ ]:
def compare_dist(features):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    ax = axes[0]
    sns.histplot(raw_df.loc[:, features], kde=True, ax=ax)
    ax.set_title("Raw features")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax = axes[1]
    sns.histplot(imputed.loc[:, features], kde=True, ax=ax)
    ax.set_title("Imputed features")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

compare_dist(["N400", "P300", "LRC", "P600", "LNC"])

## Outliers

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
sns.boxplot(data=raw_df.loc[:, ["P300", "P600", "N400", "LRC", "LNC"]], ax=ax)
ax.set_ylabel("Amplitude")
ax.set_title("Boxplots utilisant le critère de Tukey")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

In [ ]:
imputed

### Colonnes utilisées pour la détection d'outliers

On définit la liste des colonnes une seule fois : variables continues + colonnes encodées Hipp_*.
Toute la suite (comparaison des méthodes, Isolation Forest) réutilise cette même liste.

In [ ]:
base_columns = ["P300", "P600", "LNC", "N400", "LRC", "IQ",
                "VCI", "PRI", "WMI", "SPI", "VBL", "VSL", "VBD", "VSD"]

columns = base_columns + hipp_columns
columns

## Comparaison des méthodes de détection d'outliers

In [ ]:
column_pairs = list(combinations(columns, 2))
outliers_fraction = 0.15

anomaly_algorithms = {
    "Robust covariance": lambda: EllipticEnvelope(contamination=outliers_fraction, random_state=42),
    "One-Class SVM": lambda: OneClassSVM(nu=outliers_fraction, kernel="rbf", gamma=0.1),
    "One-Class SVM (SGD)": lambda: make_pipeline(
        Nystroem(gamma=0.1, random_state=42, n_components=150),
        SGDOneClassSVM(nu=outliers_fraction, shuffle=True, fit_intercept=True,
                        random_state=42, tol=1e-6),
    ),
    "Isolation Forest": lambda: IsolationForest(contamination=outliers_fraction, random_state=42),
    "Local Outlier Factor": lambda: LocalOutlierFactor(n_neighbors=35, contamination=outliers_fraction),
}

agreement_scores = {name: [] for name in anomaly_algorithms}
fraction_detected = {name: [] for name in anomaly_algorithms}
timings = {name: [] for name in anomaly_algorithms}
failed_pairs = {name: [] for name in anomaly_algorithms}

for col_x, col_y in column_pairs:
    X = imputed[[col_x, col_y]].to_numpy()

    if np.isnan(X).any():
        continue

    predictions = {}

    for name, make_algo in anomaly_algorithms.items():
        algo = make_algo()
        try:
            t0 = time.time()
            if name == "Local Outlier Factor":
                y_pred = algo.fit_predict(X)
            else:
                y_pred = algo.fit(X).predict(X)
            t1 = time.time()

            predictions[name] = y_pred
            timings[name].append(t1 - t0)
            fraction_detected[name].append(np.mean(y_pred == -1))
        except Exception:
            failed_pairs[name].append((col_x, col_y))
            predictions[name] = None

    valid_preds = {k: v for k, v in predictions.items() if v is not None}
    if len(valid_preds) < 2:
        continue

    stacked = np.stack(list(valid_preds.values()))
    consensus = np.sign(np.sum(stacked, axis=0))
    consensus[consensus == 0] = 1

    for name, y_pred in valid_preds.items():
        agreement = np.mean(y_pred == consensus)
        agreement_scores[name].append(agreement)

summary = []
for name in anomaly_algorithms:
    mean_agreement = np.mean(agreement_scores[name]) if agreement_scores[name] else np.nan
    mean_fraction = np.mean(fraction_detected[name]) if fraction_detected[name] else np.nan
    std_fraction = np.std(fraction_detected[name]) if fraction_detected[name] else np.nan
    mean_time = np.mean(timings[name]) if timings[name] else np.nan
    n_failed = len(failed_pairs[name])

    summary.append({
        "Méthode": name,
        "Accord moyen avec consensus": round(mean_agreement, 3),
        "Fraction outliers détectés (moy)": round(mean_fraction, 3),
        "Fraction outliers (écart-type)": round(std_fraction, 3),
        "Temps moyen (s)": round(mean_time, 4),
        "Paires échouées": n_failed,
    })

summary_df = pd.DataFrame(summary).sort_values(
    "Accord moyen avec consensus", ascending=False
).reset_index(drop=True)

summary_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(summary_df["Méthode"], summary_df["Accord moyen avec consensus"], color="#377eb8")
axes[0].set_xlabel("Accord moyen avec le consensus")
axes[0].set_xlim(0, 1)
axes[0].set_title("Fiabilité (accord inter-méthodes)")
axes[0].invert_yaxis()
for i, v in enumerate(summary_df["Accord moyen avec consensus"]):
    axes[0].text(v + 0.01, i, f"{v:.3f}", va="center")

axes[1].barh(
    summary_df["Méthode"],
    summary_df["Fraction outliers détectés (moy)"],
    xerr=summary_df["Fraction outliers (écart-type)"],
    color="#ff7f00",
    capsize=4
)
axes[1].axvline(outliers_fraction, color="black", linestyle="--", label="Cible (15%)")
axes[1].set_xlabel("Fraction d'outliers détectés")
axes[1].set_title("Stabilité de détection (± écart-type)")
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

**Conclusion : on retient l'Isolation Forest** pour la suite de l'analyse.

### Choix de contamination pour l'Isolation Forest

In [ ]:
X = imputed[columns].to_numpy()

clf = IsolationForest(random_state=42, contamination="auto")
clf.fit(X)
scores = clf.decision_function(X)

plt.hist(scores, bins=30)
plt.xlabel("Score d'anomalie (decision_function)")
plt.ylabel("Nombre de patients")
plt.title("Distribution des scores — chercher une cassure naturelle")
plt.show()

print(np.sort(scores)[:15])

In [ ]:
threshold = -0.02
n_outliers_visual = np.sum(scores < threshold)
contamination_estimated = n_outliers_visual / len(scores)

print(f"Nombre d'outliers estimé visuellement : {n_outliers_visual}")
print(f"Contamination estimée : {contamination_estimated:.3f}")

## Figure finale : Isolation Forest sur toutes les paires de colonnes (Hipp inclus)

In [ ]:
n_pairs = len(column_pairs)
n_cols = int(np.ceil(np.sqrt(n_pairs)))
n_rows = int(np.ceil(n_pairs / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
axes = axes.flatten()

colors = np.array(["#377eb8", "#ff7f00"])

for i, (col_x, col_y) in enumerate(column_pairs):
    ax = axes[i]

    X_pair = imputed[[col_x, col_y]].to_numpy()

    if np.isnan(X_pair).any():
        ax.set_title(f"{col_x} vs {col_y}\n(NaN, skip)", fontsize=8, color="red")
        ax.set_xticks(())
        ax.set_yticks(())
        continue

    clf = IsolationForest(contamination=contamination_estimated, random_state=42)
    y_pred = clf.fit_predict(X_pair)

    x_min, x_max = X_pair[:, 0].min() - 1, X_pair[:, 0].max() + 1
    y_min, y_max = X_pair[:, 1].min() - 1, X_pair[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contour(xx, yy, Z, levels=[0], linewidths=1.5, colors="black")
    ax.scatter(X_pair[:, 0], X_pair[:, 1], s=8, color=colors[(y_pred + 1) // 2])

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_title(f"{col_x} vs {col_y}", fontsize=9)
    ax.set_xticks(())
    ax.set_yticks(())

for j in range(n_pairs, len(axes)):
    axes[j].axis("off")

fig.suptitle("Détection d'outliers — Isolation Forest (toutes les paires, Hipp inclus)", fontsize=16, y=1.0)
plt.tight_layout()
plt.savefig("isolation_forest_all_pairs.png", dpi=150, bbox_inches="tight")
plt.show()